# 01 — Filter significant splicing events

This notebook filters SUPPA's differential splicing results to find
significant events, and then identifies which events are unique to
variable boundary mode compared to strict.

**Filter criteria:** p-value < 0.05 AND |dPSI| ≥ 0.1
**Events analysed:** A3, A5, RI
**Comparisons:** CT8 vs CT20 and CT20 vs PerKO
**Runs compared:** Strict vs Variable 1nt and Variable 5nt

In [6]:
import pandas as pd
import os

# Diff output paths
STRICT = "/Users/gricey/Desktop/Internship/data/output_strict/diff"
VAR1   = "/Users/gricey/Desktop/Internship/data/output_variable_1nt/diff"
VAR5   = "/Users/gricey/Desktop/Internship/data/output_variable_5nt/diff"

# IOE paths
IOE_STRICT = "/Users/gricey/Desktop/Internship/data/output_strict/events"
IOE_VAR1   = "/Users/gricey/Desktop/Internship/data/output_variable_1nt/events"
IOE_VAR5   = "/Users/gricey/Desktop/Internship/data/output_variable_5nt/events"

# Events and comparison
EVENTS = ["A3", "A5", "RI"]
COMPARISONS = {"CT8_vs_CT20": 0, "CT20_vs_PerKO": 1}

print("Paths and parameters defined!")

Paths and parameters defined!


## Step 1 — Apply significance filters

We filter each diff file keeping only events where:
- p-value < 0.05 (statistically significant)
- |dPSI| ≥ 0.1 (biologically meaningful change, at least 10%)

In [7]:
def load_and_filter(diff_dir, event, temp=0):
    path = f"{diff_dir}/diff_{event}.dpsi.temp.{temp}"
    df = pd.read_csv(path, sep="\t")
    df.columns = ["Event_id", "dPSI", "pval"]
    filtered = df[
        (df["pval"] < 0.05) &
        (df["dPSI"].abs() >= 0.1)
    ].copy()
    return filtered

# Apply to all events, runs and comparisons
results = {}
for comp_name, temp in COMPARISONS.items():
    results[comp_name] = {}
    for event in EVENTS:
        results[comp_name][event] = {
            "strict": load_and_filter(STRICT, event, temp),
            "var1":   load_and_filter(VAR1,   event, temp),
            "var5":   load_and_filter(VAR5,   event, temp),
        }

print("Filtering done!")

Filtering done!


## Step 2 — Summary table of significant event counts

Quick overview of how many events pass the filter per run and comparison.
This is the numerical reference before we look at which specific events differ.

In [8]:
rows = []
for comp_name in COMPARISONS:
    for event in EVENTS:
        rows.append({
            "Comparison": comp_name,
            "Event": event,
            "Strict": len(results[comp_name][event]["strict"]),
            "Variable 1nt": len(results[comp_name][event]["var1"]),
            "Variable 5nt": len(results[comp_name][event]["var5"]),
        })

df_summary = pd.DataFrame(rows)
df_summary["V1 - S"] = df_summary["Variable 1nt"] - df_summary["Strict"]
df_summary["V5 - S"] = df_summary["Variable 5nt"] - df_summary["Strict"]

display(df_summary)

,Comparison,Event,Strict,Variable 1nt,Variable 5nt,V1 - S,V5 - S
0,CT8_vs_CT20,A3,36,73,75,37,39
1,CT8_vs_CT20,A5,35,63,60,28,25
2,CT8_vs_CT20,RI,11,62,61,51,50
3,CT20_vs_PerKO,A3,45,73,74,28,29
4,CT20_vs_PerKO,A5,47,76,72,29,25
5,CT20_vs_PerKO,RI,13,64,65,51,52


## Step 3 — Find events unique to variable mode

The numerical subtraction above tells us *how many* extra events variable
mode finds. Now we find *which specific events* those are — events present
in variable but completely absent in strict.

This is done using a set difference on the Event_id column.

In [9]:
def unique_to_variable(var_df, strict_df):
    strict_ids = set(strict_df["Event_id"])
    unique = var_df[~var_df["Event_id"].isin(strict_ids)].copy()
    return unique

unique_events = {}
for comp_name in COMPARISONS:
    unique_events[comp_name] = {}
    for event in EVENTS:
        unique_events[comp_name][event] = {
            "var1": unique_to_variable(
                results[comp_name][event]["var1"],
                results[comp_name][event]["strict"]
            ),
            "var5": unique_to_variable(
                results[comp_name][event]["var5"],
                results[comp_name][event]["strict"]
            ),
        }

# Print summary
for comp_name in COMPARISONS:
    print(f"\n{comp_name}:")
    for event in EVENTS:
        n1 = len(unique_events[comp_name][event]["var1"])
        n5 = len(unique_events[comp_name][event]["var5"])
        print(f"  {event}: {n1} unique to Var1nt, {n5} unique to Var5nt")


CT8_vs_CT20:
  A3: 73 unique to Var1nt, 75 unique to Var5nt
  A5: 63 unique to Var1nt, 60 unique to Var5nt
  RI: 62 unique to Var1nt, 61 unique to Var5nt

CT20_vs_PerKO:
  A3: 73 unique to Var1nt, 74 unique to Var5nt
  A5: 76 unique to Var1nt, 72 unique to Var5nt
  RI: 64 unique to Var1nt, 65 unique to Var5nt


## Step 4 — Output unique events to IOE files

We now look up these unique Event_ids in the original IOE files to get
their full IOE entries, and save them to new IOE files for downstream
analysis in IGV.

In [10]:
IOE_SUFFIXES = {
    "var1": ("variable_1", IOE_VAR1),
    "var5": ("variable_5", IOE_VAR5),
}

OUTPUT_DIR = "/Users/gricey/Desktop/Internship/data/ioe_diff"
os.makedirs(OUTPUT_DIR, exist_ok=True)

for comp_name in COMPARISONS:
    for event in EVENTS:
        for var_key, (suffix, ioe_dir) in IOE_SUFFIXES.items():
            ioe_path = f"{ioe_dir}/events_{event}_{suffix}.ioe"
            ioe_df = pd.read_csv(ioe_path, sep="\t")

            unique_ids = set(unique_events[comp_name][event][var_key]["Event_id"])
            ioe_unique = ioe_df[ioe_df["event_id"].isin(unique_ids)]

            out_path = f"{OUTPUT_DIR}/{comp_name}_{event}_{var_key}_unique.ioe"
            ioe_unique.to_csv(out_path, sep="\t", index=False)
            print(f"Saved {len(ioe_unique)} events → {out_path}")

Saved 73 events → /Users/gricey/Desktop/Internship/data/ioe_diff/CT8_vs_CT20_A3_var1_unique.ioe
Saved 75 events → /Users/gricey/Desktop/Internship/data/ioe_diff/CT8_vs_CT20_A3_var5_unique.ioe
Saved 63 events → /Users/gricey/Desktop/Internship/data/ioe_diff/CT8_vs_CT20_A5_var1_unique.ioe
Saved 60 events → /Users/gricey/Desktop/Internship/data/ioe_diff/CT8_vs_CT20_A5_var5_unique.ioe
Saved 62 events → /Users/gricey/Desktop/Internship/data/ioe_diff/CT8_vs_CT20_RI_var1_unique.ioe
Saved 61 events → /Users/gricey/Desktop/Internship/data/ioe_diff/CT8_vs_CT20_RI_var5_unique.ioe
Saved 73 events → /Users/gricey/Desktop/Internship/data/ioe_diff/CT20_vs_PerKO_A3_var1_unique.ioe
Saved 74 events → /Users/gricey/Desktop/Internship/data/ioe_diff/CT20_vs_PerKO_A3_var5_unique.ioe
Saved 76 events → /Users/gricey/Desktop/Internship/data/ioe_diff/CT20_vs_PerKO_A5_var1_unique.ioe
Saved 72 events → /Users/gricey/Desktop/Internship/data/ioe_diff/CT20_vs_PerKO_A5_var5_unique.ioe
Saved 64 events → /Users/gricey/